# LangChain: Models, Prompts and Output Parsers (LangChain：模型、提示词和输出解析器)


## Outline (大纲)

 * Direct API calls to OpenAI (直接调用 OpenAI API)
 * API calls through LangChain: (通过 LangChain 调用 API：)
   * Prompts (提示词 (Prompts)：如何结构化地引导模型。)
   * Models (模型 (Models)：切换不同的 LLM 背景。)
   * Output parsers (输出解析器 (Output Parsers)：将模型的文本回复转化为结构化数据（如 JSON）。)

## Get your [OpenAI API Key](https://platform.openai.com/account/api-keys)

In [ ]:
#!pip install python-dotenv
#!pip install openai

In [48]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

load_dotenv(find_dotenv())

# 创建客户端（指向通义千问 API）
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),  # 从环境变量读取
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the video.

注意：LLM 算法并非总能产生相同的结果。在笔记本中运行代码时，您可能会得到与视频中略有不同的结果。

本代码用千问模型，下面代码不需要执行

In [2]:
# account for deprecation of LLM model
import datetime
# Get the current date
current_date = datetime.datetime.now().date()

# Define the date after which the model should be set to "gpt-3.5-turbo"
target_date = datetime.date(2024, 6, 12)

# Set the model variable based on the current date
if current_date > target_date:
    llm_model = "gpt-3.5-turbo"
else:
    llm_model = "gpt-3.5-turbo-0301"

## Chat API : OpenAI

Let's start with a direct API calls to OpenAI.

In [51]:
llm_model = "qwen-max"
def get_completion(prompt, model=llm_model):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content

In [52]:
get_completion("What is 1+1?")

'1+1 equals 2.'

In [53]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [54]:
style = """American English \
in a calm and respectful tone
"""

In [55]:
prompt = f"""Translate the text \
that is delimited by triple backticks 
into a style that is {style}.
text: ```{customer_email}```
"""

print(prompt)

Translate the text that is delimited by triple backticks 
into a style that is American English in a calm and respectful tone
.
text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse,the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



In [56]:
response = get_completion(prompt)

In [57]:
response

'Sure, here is the translation into a calm and respectful American English style:\n\n"Hello, I\'m quite frustrated that the lid of my blender came off and splattered my kitchen walls with smoothie. To make matters worse, the warranty doesn\'t cover the cost of cleaning up my kitchen. I could really use your help right now."'

## Chat API : LangChain

Let's try how we can do the same using LangChain.

### Model

In [11]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 47.5 kB/s  0:00:156m-:--:--
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.0
    Uninstalling langchain-core-1.3.0:
      Successfully uninstalled langchain-core-1.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [langchain_openai][langchain-core]

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [58]:
# 新版本导入路径变了
from langchain_openai import ChatOpenAI

In [59]:
# To control the randomness and creativity of the generated
# text by an LLM, use temperature = 0.0
chat = ChatOpenAI(
    temperature=0.0, 
    model=llm_model,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY")
    )
chat

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x148553d90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x106a62f10>, root_client=<openai.OpenAI object at 0x10ac32d50>, root_async_client=<openai.AsyncOpenAI object at 0x148553a10>, model_name='qwen-max', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://dashscope.aliyuncs.com/compatible-mode/v1', stream_chunk_timeout=120.0)

### Prompt template

In [60]:
# 定义带变量的提示词模板（{style}和{text}是占位符）
# 翻译如下：
# 将由三反引号分隔的文本翻译成 {style} 样式。
# 文本： '''{text}'''
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

In [61]:
# 从 langchain_core 的 prompts 模块中导入 ChatPromptTemplate 类
# 该类专门用于处理“聊天式”提示词（包含 System, AI, Human 等角色）的模板化
from langchain_core.prompts import ChatPromptTemplate

# 注释：说明下方的操作是利用 LangChain 的工具来创建一个可复用的模板对象
# 用LangChain创建模板对象
# 
# 使用 ChatPromptTemplate 类的静态方法 from_template 根据传入的字符串创建一个模板实例。
# template_string 通常包含像 {topic} 这样的占位符，LangChain 会自动解析这些变量，
# 方便后续通过调用 prompt_template.format(topic="xxx") 来动态填充内容。
prompt_template = ChatPromptTemplate.from_template(template_string)

In [62]:
# prompt_template: 之前通过 from_template 创建的 ChatPromptTemplate 顶层对象
# .messages: 访问该模板内部的消息列表（List），列表中每个元素代表一条消息（如 System, Human, AI）
# [0]: 索引到列表中的第一个元素。由于你是通过 from_template 创建的，这里通常是一个 HumanMessagePromptTemplate 对象
# .prompt: 获取该消息对象内部封装的底层 PromptTemplate 实例，它包含了原始的字符串模板和解析出的变量列表
prompt_template.messages[0].prompt

PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{text}```\n')

In [63]:
prompt_template.messages[0].prompt.input_variables

['style', 'text']

In [64]:
customer_style = """American English \
in a calm and respectful tone
"""

In [65]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [68]:
customer_messages = prompt_template.format_messages(
                    style=customer_style,
                    text=customer_email)

In [69]:
print(type(customer_messages))
print(type(customer_messages[0]))

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>


In [70]:
print(customer_messages[0])

content="Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone\n. text: ```\nArrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n" additional_kwargs={} response_metadata={}


In [71]:
# Call the LLM to translate to the style of the customer message
# 调用LangChain封装的模型
customer_response = chat.invoke(customer_messages)

In [72]:
print(customer_response.content)

Sure, here's the translation into a calm and respectful American English style:

"Hello, I am quite upset that the lid of my blender came off and splattered smoothie all over my kitchen walls. To make matters worse, the warranty does not cover the cost of cleaning up the mess. I could really use your help with this, please."


In [73]:
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

In [74]:
service_style_pirate = """\
a polite tone \
that speaks in English Pirate\
"""

In [75]:
service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply)

print(service_messages[0].content)

Translate the text that is delimited by triple backticks into a style that is a polite tone that speaks in English Pirate. text: ```Hey there customer, the warranty does not cover cleaning expenses for your kitchen because it's your fault that you misused your blender by forgetting to put the lid on before starting the blender. Tough luck! See ya!
```



In [76]:
service_response = chat.invoke(service_messages)
print(service_response.content)

Arrr, me hearty! Greetings to ye, fine customer. It be comin' t' me notice that yer warranty, as it stands, doth not cover the cleanin' o' yer kitchen. Ye see, 'twas a wee bit o' a misstep on yer part, forgettin' t' put the lid on yer blender afore givin' it full sail. Aye, 'tis a tough spot ye find yerself in, but such be the way o' the sea. Farewell, and may fair winds guide ye henceforth!


## Output Parsers （输出解析器）

Let's start with defining how we would like the LLM output to look like:

首先，让我们来定义一下我们希望 LLM 输出结果是什么样子：

In [77]:
{
  "gift": False,
  "delivery_days": 5,
  "price_value": "pretty affordable!"
}

{'gift': False, 'delivery_days': 5, 'price_value': 'pretty affordable!'}

In [78]:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [80]:
from langchain_core.prompts import ChatPromptTemplate

# 用LangChain创建模板对象
prompt_template = ChatPromptTemplate.from_template(review_template)
print(prompt_template)

input_variables=['text'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\nFormat the output as JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: {text}\n'), additional_kwargs={})]


In [81]:
messages = prompt_template.format_messages(text=customer_review)
response = chat.invoke(messages)
print(response.content)

```json
{
  "gift": true,
  "delivery_days": 2,
  "price_value": ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]
}
```


In [82]:
type(response.content)

str

In [83]:
# You will get an error by running this line of code  运行这行代码会出错
# because'gift' is not a dictionary 因为'gift'不是字典
# 'gift' is a string  'gift' 是一个字符串
response.content.get('gift')

AttributeError: 'str' object has no attribute 'get'

### Parse the LLM output string into a Python dictionary（将 LLM 输出字符串解析为 Python 字典）

新版本有很大的变化，下面是按照新版本改写后的

In [84]:
from pydantic import BaseModel, Field, conint
from typing import Literal, List, Annotated

# 定义输出结构，比 ResponseSchema 更强大、类型更严格
class ReviewAnalysis(BaseModel):
    """从产品评论中提取关键信息的结构。"""
    
    # gift: True/False
    gift: bool = Field(
        description="Was the item purchased\
                    as a gift for someone else? \
                    Answer True if yes,\
                    False if not or unknown."
    )
    
    # delivery_days: 整数，如果找不到信息则为 -1
    # conint(ge=-1) 确保值大于等于 -1
    delivery_days: int = Field(
        description="How many days\
                    did it take for the product\
                    to arrive? If this \
                    information is not found,\
                    output -1.",
        ge=-1 # 使用 Field 的 ge（大于等于）参数进行约束
    )
    
    # price_value: 句子列表
    price_value: List[str] = Field(
        description="Extract any\
                                    sentences about the value or \
                                    price, and output them as a \
                                    comma separated Python list."
    )

In [85]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


# 这会自动告诉 LLM 以 JSON 格式返回符合 ReviewAnalysis 结构的数据
structured_llm = chat.with_structured_output(ReviewAnalysis)

result: ReviewAnalysis = structured_llm.invoke(messages)

# 结果可以直接作为 Python 对象访问
print(result)
print(f"\n是否为礼物: {result.gift}")
print(f"到货天数: {result.delivery_days}")
print(f"价格: {result.price_value}")

gift=True delivery_days=2 price_value=["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]

是否为礼物: True
到货天数: 2
价格: ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]


### 老版本实现
为了方便比对，老版本代码保留，但是无法运行，会报错，下方表格为两种方法对比

|**特性**|**旧方法 (v0.x) - StructuredOutputParser**|**新方法 (v1.0+) - Pydantic + 函数调用**|
|---|---|---|
|**何时定义结构**|在 LLM 返回原始文本后，用代码工具进行转换。|**在执行前**，将 Pydantic 结构定义传递给模型 API。|
|**工作原理**|依靠 LLM 输出**格式正确的 JSON 字符串** → Python 代码解析。|依靠 LLM **底层 API 的能力**（如 OpenAI 的 Function Calling）→ **强制** LLM 返回符合该结构的 JSON。|
|**可靠性**|较低，如果 LLM 稍微出错（多余的字符、引号错误），解析就会失败。|极高，API 会确保输出是有效的 JSON，并且格式符合要求。|
|**核心挑战**|**解析（Parsing）**|**指令（Instruction）**|


In [43]:
from langchain_core.output_parsers import ResponseSchema
from langchain_core.output_parsers import StructuredOutputParser

ImportError: cannot import name 'ResponseSchema' from 'langchain_core.output_parsers' (c:\Users\bangsun\miniconda3\envs\jupterlab\Lib\site-packages\langchain_core\output_parsers\__init__.py)

In [ ]:
gift_schema = ResponseSchema(name="gift",
                             description="Was the item purchased\
                             as a gift for someone else? \
                             Answer True if yes,\
                             False if not or unknown.")
delivery_days_schema = ResponseSchema(name="delivery_days",
                                      description="How many days\
                                      did it take for the product\
                                      to arrive? If this \
                                      information is not found,\
                                      output -1.")
price_value_schema = ResponseSchema(name="price_value",
                                    description="Extract any\
                                    sentences about the value or \
                                    price, and output them as a \
                                    comma separated Python list.")

response_schemas = [gift_schema, 
                    delivery_days_schema,
                    price_value_schema]

In [ ]:
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

In [ ]:
format_instructions = output_parser.get_format_instructions()

In [ ]:
print(format_instructions)

In [ ]:
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

text: {text}

{format_instructions}
"""

prompt = ChatPromptTemplate.from_template(template=review_template_2)

messages = prompt.format_messages(text=customer_review, 
                                format_instructions=format_instructions)

In [ ]:
print(messages[0].content)

In [ ]:
response = chat.invoke(messages)

In [ ]:
print(response.content)

In [ ]:
output_dict = output_parser.parse(response.content)

In [ ]:
output_dict

In [ ]:
type(output_dict)

In [ ]:
output_dict.get('delivery_days')

Reminder: Download your notebook to you local computer to save your work.